Da inserire in project se si vuole la demo per fare retrieval delle immagini usando la mia implementazione

In [ ]:
import gradio as gr

def gradio_retrieval_interface(query_id, modifiers_str, beta, pool_size_N):
    """Funzione ponte tra l'interfaccia Gradio e i tuoi algoritmi."""
    try:
        # 1. Convertiamo l'input dell'utente
        id_query = int(query_id)
        # Puliamo la stringa degli attributi (es: "+ Eyeglasses, - Smiling" -> ['+Eyeglasses', '-Smiling'])
        attributes = [attr.strip() for attr in modifiers_str.split(",") if attr.strip()]

        # 2. Estraiamo l'immagine di query originale e il suo embedding
        # Assumiamo che 'celeba' sia accessibile globalmente nel tuo notebook
        query_image = celeba[id_query][0]
        query_emb = image_embeddings[id_query]

        # 3. Chiamiamo la tua funzione CH_mixed_cond_retrieval
        retrieved_indices, _ = CH_mixed_cond_retrieval(
            query_img_emb=query_emb,
            id_query=id_query,
            database_embeddings=image_embeddings,
            attributes=attributes,
            ordered_attributes=ordered_attributes,  # Assunta globale
            model_head=model_head,  # Assunto globale
            top_k=10,  # Ne chiediamo 10 per fare le due linee
            device=device,  # Assunta globale (es. "cuda" o "cpu")
            N=int(pool_size_N),
            beta=beta,
        )

        # 4. Generiamo il plot personalizzato a due righe (riadattato per Gradio)
        fig = plt.figure(figsize=(14, 9))

        # Subplot Query
        ax_query = plt.subplot2grid((3, 5), (0, 2))
        ax_query.imshow(query_image)
        ax_query.axis("off")
        ax_query.set_title("YOUR QUERY IMAGE", fontweight="bold", fontsize=11)

        # Stampa dei risultati su due linee
        num_results = min(10, len(retrieved_indices))
        for i in range(num_results):
            db_idx = retrieved_indices[i]
            retrieved_img = celeba[db_idx][0]

            row = 1 if i < 5 else 2
            col = i % 5

            ax_res = plt.subplot2grid((3, 5), (row, col))
            ax_res.imshow(retrieved_img)
            ax_res.axis("off")
            ax_res.set_title(f"Top-{i+1} (ID: {db_idx})", fontsize=9)

        fig.subplots_adjust(hspace=0.5, wspace=0.2)

        # Importante: Gradio accetta un oggetto Figure di Matplotlib in output
        return fig

    except Exception as e:
        # In caso di errore (es. ID non valido), mostra un plot vuoto con l'errore scritto
        fig, ax = plt.subplots(figsize=(6, 2))
        ax.text(
            0.5,
            0.5,
            f"Errore: {str(e)}\nControlla che l'ID sia corretto.",
            ha="center",
            va="center",
            color="red",
        )
        ax.axis("off")
        return fig


# --- COSTRUZIONE DELL'INTERFACCIA GRAFICA (Gradio Blocks) ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🚀 Conditionally-Headed Image Retrieval Demo")
    gr.Markdown(
        "Inserisci un'immagine di query da CelebA e applica modificatori testuali per guidare il re-ranking tramite la testa di classificazione."
    )

    with gr.Row():
        # Colonna di Sinistra: Input dei comandi
        with gr.Column(scale=1):
            query_id_input = gr.Textbox(
                value="0",
                label="Query Image ID (CelebA Index)",
                placeholder="Es: 1250",
            )

            modifiers_input = gr.Textbox(
                value="+Eyeglasses, -Smiling",
                label="Attribute Modifiers",
                placeholder="Usa + per aggiungere e - per togliere, separati da virgola",
            )

            beta_slider = gr.Slider(
                minimum=0.0,
                maximum=1.0,
                value=0.5,
                step=0.05,
                label="Beta (Weight: CLIP vs Classification Head)",
                info="0 = Solo CLIP | 1 = Solo Head Classificatore",
            )

            n_slider = gr.Slider(
                minimum=50,
                maximum=2000,
                value=610,
                step=50,
                label="Pool Size (N)",
                info="Quanti candidati estrarre da CLIP prima del re-ranking",
            )

            submit_btn = gr.Button("🔍 Esegui Retrieval", variant="primary")

        # Colonna di Destra: Output visivo (La griglia dei grafici)
        with gr.Column(scale=2):
            plot_output = gr.Plot(label="Retrieval Results (Top 1-10)")

    # Definizione del trigger del pulsante
    submit_btn.click(
        fn=gradio_retrieval_interface,
        inputs=[query_id_input, modifiers_input, beta_slider, n_slider],
        outputs=plot_output,
    )

# Avvia la demo localmente
demo.launch(debug=True, share=True)